# 코사인 유사도를 이용한 추천 시스템

## Numpy를 이용해서 코사인 유사도를 계산하는 함수를 구현

### 예시 문서
문서 1 : 저는 사과 좋아요  
문서 2 : 저는 바나나 좋아요  
문서 3 : 저는 바나나 좋아요 저는 바나나 좋아요  
  
띄어쓰기 기준 토큰화  
||바나나|사과|저는|좋아요|
|--|--|--|--|--|
|문서 1|0|1|1|1|
|문서 2|1|0|1|1|
|문서 3|2|0|2|2|

In [1]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

doc1 = np.array([0, 1, 1, 1])
doc2 = np.array([1, 0, 1, 1])
doc3 = np.array([2, 0, 2, 2])

print('문서 1과 문서 2의 코사인 유사도:', cos_sim(doc1, doc2))
print('문서 1과 문서 3의 코사인 유사도:', cos_sim(doc1, doc3))
print('문서 2과 문서 3의 코사인 유사도:', cos_sim(doc2, doc3))

문서 1과 문서 2의 코사인 유사도: 0.6666666666666667
문서 1과 문서 3의 코사인 유사도: 0.6666666666666667
문서 2과 문서 3의 코사인 유사도: 1.0000000000000002


## 유사도를 이용한 추천 시스템 구현하기

영화를 입력하면 해당 영화의 줄거리와 유사한 줄거리의 영화를 찾아서 추천하는 시스템

In [4]:
import kagglehub

path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

100%|██████████| 228M/228M [00:01<00:00, 191MB/s] 

Extracting files...


In [12]:
!ls {path}

credits.csv   links.csv        movies_metadata.csv  ratings_small.csv
keywords.csv  links_small.csv  ratings.csv


In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = pd.read_csv(path + "/movies_metadata.csv", low_memory=False)
data.head(2)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [ ]:
#data = data.head(20000)

In [15]:
print('overview 열의 결측값 개수:', data['overview'].isnull().sum())

overview 열의 결측값 개수: 135


In [16]:
data['overview'] = data['overview'].fillna('')

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['overview'])
print('TF-IDF 행렬의 크기:', tfidf_matrix.shape)
#20000개의 영화와 47847개의 단어가 사용됨

TF-IDF 행렬의 크기: (20000, 47487)


In [22]:
title_to_index = dict(zip(data['title'], data.index))
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [24]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = title_to_index[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return data['title'].iloc[movie_indices]

In [ ]:
get_recommendations('The Dark Knight Rises')